In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
project_root = os.path.abspath("")
if project_root not in sys.path:
    sys.path.append(project_root)
import sleap
from pathlib import Path
from ipywidgets import widgets
from IPython.display import display
import matplotlib.pyplot as plt
from hypnose_analysis.utils.visualization_utils import _get_from_cache, _update_cache
from sleap_utils import *

%matplotlib widget

INFO:numexpr.utils:Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
# Wrapper function to process all .slp files. 
# Params:
    # subjid: list of subjids to process (e.g., [40, 41, 42])
    # date: specific date or list of dates to process (e.g., [20251030, 20251231]) or None for all dates
    # core_nodes: list of SLEAP nodes used to calculate centroid (default uses most reliable nodes)
    # recompute: if True, recomputes and overwrites existing output files. Keep False unless SLEAP model has been updated. 
    # base_dir: optional to set to different base dir and override symlinked default path
sleap_processing = process_sleap_sessions(
    subjid=[40],
    date=[20251128],
    base_dir=None,
    core_nodes=None, 
    save_output=True, 
    recompute=True
)

Found 5 video(s) to process:
  1. 2025-11-28T15-25-28__VideoData_1904-01-03T03-00-00.predictions.slp
  2. 2025-11-28T15-25-28__VideoData_1904-01-03T04-00-00.predictions.slp
  3. 2025-11-28T15-25-28__VideoData_1904-01-03T05-00-00.predictions.slp
  4. 2025-11-28T17-12-32__VideoData_1904-01-03T05-00-00.predictions.slp
  5. 2025-11-28T17-58-30__VideoData_1904-01-03T06-00-00.predictions.slp

[1/5] Processing: 2025-11-28T15-25-28__VideoData_1904-01-03T03-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video1_2025-11-28T15-25-28__VideoData_1904-01-03T03-00-00.csv
    Total rows: 50428
    Frame range: 2161 to 59425

[2/5] Processing: 2025-11-28T15-25-28__VideoData_1904-01-03T04-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video2_2025-11-28T15-25-28__VideoData_1904-01-03T04-00-00.csv
    Total rows: 194600
    Frame range: 0 to 216007

[3/5] Processing: 2025-11-28T15-25-28__VideoData_1904-01-03T05-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video3_2025-11-28T15-25-28__VideoData

## Step by Step .slp file processing (use above wrapper for multiple subjids/dates)

In [13]:
# 1. Step: extract frames from .slp files and save with centroid coordinates in csv files per video

subjid = 40
date = 20251030

sleap_data = sleap_labels_and_centroid(subjid=subjid, date=date, skip_empty=True)

Found 4 video(s) to process:
  1. VideoData_1904-01-01T02-00-00.predictions.slp
  2. VideoData_1904-01-01T03-00-00.predictions.slp
  3. VideoData_1904-01-01T04-00-00.predictions.slp
  4. VideoData_1904-01-01T05-00-00.predictions.slp

[1/4] Processing: VideoData_1904-01-01T02-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video1.csv
    Total rows: 33311
    Frame range: 566 to 43849

[2/4] Processing: VideoData_1904-01-01T03-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video2.csv
    Total rows: 94856
    Frame range: 0 to 157337

[3/4] Processing: VideoData_1904-01-01T04-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video3.csv
    Total rows: 42167
    Frame range: 123904 to 182857

[4/4] Processing: VideoData_1904-01-01T05-00-00.predictions.slp
  ⚠️ No pose data found in \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\derivatives\sub-040_id-259\ses-003_date-20251030\saved_analysis_results\VideoData_1904-01-01T05-00-00.predictions.slp, skipping this file

✅ All videos proc

In [ ]:
# 2. Step: add timestamps to all video tracking csv files and combine into single dataframe
tracking_times = add_timestamps_to_sleap_tracking(subjid=40, date=20251030)

Found 3 SLEAP tracking file(s)
Found 1 experiment folder(s)
Could not create odour_led (missing output data)
  Loaded 579262 frames from experiment 0
Total: 579,262 frames from 4 video file(s)

Video files in order:
  1. VideoData_1904-01-01T02-00-00.avi
  2. VideoData_1904-01-01T03-00-00.avi
  3. VideoData_1904-01-01T04-00-00.avi
  4. VideoData_1904-01-01T05-00-00.avi
Matched sleap_tracking_video1.csv (video 1) to VideoData_1904-01-01T02-00-00.avi
Matched sleap_tracking_video2.csv (video 2) to VideoData_1904-01-01T03-00-00.avi
Matched sleap_tracking_video3.csv (video 3) to VideoData_1904-01-01T04-00-00.avi

Processing sleap_tracking_video1.csv:
  SLEAP frames: 566 - 43849 (33311 rows)
  Video local frames: 0 - 43849
  Matched 33311/33311 frames to timestamps

Processing sleap_tracking_video2.csv:
  SLEAP frames: 0 - 157337 (94856 rows)
  Video local frames: 0 - 216007
  Matched 94856/94856 frames to timestamps

Processing sleap_tracking_video3.csv:
  SLEAP frames: 123904 - 182857 (421

## Video Creation: Create an annotated video with centroid overlay and odor presentation + reward status information

In [13]:
output_video = annotate_videos_with_sleap_and_trials(
    subjid=40,
    date=20251128,
    rotate_deg=90,
    base_dir="Z:/hypnose",
    time_window=("00:03:16", "00:03:47"), 
    video_indices=[2], # which videos to annotate, can be multiple by passing [1, 2, 3] or all by passing None
    reward_display_s=2,
    mark_timepoint="15:45:22.563"
)
# good example video quintuples: 40, 20251120, index 2, 0:10:38 to 0:11:10 (3 trials, B, A, B)

Loaded combined timestamps from cache: 479565 frames

Processing video 1/1 (original #2): 2025-11-28T15-25-28__VideoData_1904-01-03T04-00-00.avi
  Trimmed to window: 0 days 00:03:16 - 0 days 00:03:47 (1860 frames)
  Found 1860 frames with timestamps
  Video: 1280x1024 @ 60.0 fps, 216008 frames
  Saving to: Z:\hypnose\derivatives\sub-040_id-259\ses-025_date-20251128\saved_analysis_results\sleap_visualization_rotdeg_90_video2.mp4


  Encoding: 100%|██████████| 1860/1860 [01:36<00:00, 19.34frames/s]

  ✓ Completed!

✅ All videos processed and saved!


# Miscellaneous 

In [9]:
# Cell to check what real time (e.g., sequence_start) relates to what video index and time from start of video, to help set parameters for annotate_videos_with_sleap_and_trials time_window and video_indices
from pathlib import Path
import pandas as pd
from math import floor, ceil
from hypnose_analysis.paths import get_derivatives_root
from hypnose_analysis.utils.visualization_utils import _get_from_cache, _update_cache

# Given absolute clock times, find which video covers that window and compute per-video offsets
subjid = 40
date = 20251128
start_time_str = "16:29:45"  # HH:MM:SS (rounded down to this second)
end_time_str   = "16:30:15"  # HH:MM:SS (rounded up to this second)

# Helper for formatting seconds (available both paths)
def _fmt_hhmmss(seconds):
    seconds = max(seconds, 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# Try cached summary to avoid re-reading CSVs
cache_kind = "sleap_timestamps"
cached = _get_from_cache(subjid, date, kind=cache_kind)
if cached is not None:
    df = cached.get("df") if isinstance(cached, dict) else cached
    combined_path = cached.get("combined_path") if isinstance(cached, dict) else None
else:
    cached = None

if cached is None or df is None:
    date_str = str(date)
    sub_str = f"sub-{subjid:03d}"
    deriv_root = get_derivatives_root()
    sub_dirs = list(deriv_root.glob(f"{sub_str}_id-*"))
    if not sub_dirs:
        raise FileNotFoundError(f"No subject dir for {sub_str} under {deriv_root}")
    ses_dirs = list(sub_dirs[0].glob(f"ses-*_date-{date_str}"))
    if not ses_dirs:
        raise FileNotFoundError(f"No session dir for date {date_str}")
    results_dir = ses_dirs[0] / "saved_analysis_results"

    combined_files = list(results_dir.glob("*_combined_sleap_tracking_timestamps.csv"))
    if not combined_files:
        raise FileNotFoundError("No combined timestamps CSV found; run add_timestamps_to_sleap_tracking first")
    combined_path = combined_files[0]

    df = pd.read_csv(combined_path)
    if "time" not in df.columns or "video_file" not in df.columns:
        raise ValueError("Combined CSV missing required columns 'time' or 'video_file'")

    # Normalize times to naive for comparison
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["time_naive"] = df["time"].dt.tz_localize(None)

    # Cache dataframe and path for reuse
    _update_cache(subjid, [date], {date: {"df": df, "combined_path": combined_path}}, kind=cache_kind)
else:
    # Cached df may be missing time_naive; ensure present
    if "time_naive" not in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df["time_naive"] = df["time"].dt.tz_localize(None)

date_str = str(date)
sub_str = f"sub-{subjid:03d}"
if 'deriv_root' not in locals():
    deriv_root = get_derivatives_root()
sub_dirs = list(deriv_root.glob(f"{sub_str}_id-*"))
if not sub_dirs:
    raise FileNotFoundError(f"No subject dir for {sub_str} under {deriv_root}")
ses_dirs = list(sub_dirs[0].glob(f"ses-*_date-{date_str}"))
if not ses_dirs:
    raise FileNotFoundError(f"No session dir for date {date_str}")
results_dir = ses_dirs[0] / "saved_analysis_results"

# Order videos by first appearance in the combined file
video_order = []
seen = set()
for vf in df["video_file"]:
    if vf not in seen:
        video_order.append(vf)
        seen.add(vf)
video_index_map = {vf: idx + 1 for idx, vf in enumerate(video_order)}  # 1-based

summary = []
for vf, sub in df.groupby("video_file"):
    t_min = sub["time_naive"].min()
    t_max = sub["time_naive"].max()
    summary.append((vf, t_min, t_max))

start_dt = pd.to_datetime(f"{date_str} {start_time_str}", errors="coerce")
end_dt = pd.to_datetime(f"{date_str} {end_time_str}", errors="coerce")
if pd.isna(start_dt) or pd.isna(end_dt):
    raise ValueError("Invalid start/end times: could not parse")
if end_dt <= start_dt:
    # Auto-swap if user accidentally reversed times
    print("Note: end_time precedes start_time; swapping them for the lookup")
    start_dt, end_dt = end_dt, start_dt

# Global recording bounds for sanity check
rec_min = df["time_naive"].min()
rec_max = df["time_naive"].max()
if start_dt < rec_min or end_dt > rec_max:
    print(f"Warning: requested window [{start_dt} .. {end_dt}] extends outside recorded range [{rec_min} .. {rec_max}]")

# Find the video that contains the start time
matches = [item for item in summary if item[1] <= start_dt <= item[2]]
if not matches:
    print("No video covers the requested start time.")
else:
    vf, t_min, t_max = matches[0]
    idx = video_index_map.get(vf, None)
    # Round start down to whole second, end up to whole second (but not past video end)
    start_offset_raw = (start_dt - t_min).total_seconds()
    end_offset_raw = (min(end_dt, t_max) - t_min).total_seconds()
    start_offset = floor(start_offset_raw)
    end_offset = ceil(end_offset_raw)
    print(f"Video file: {vf}")
    if idx is not None:
        print(f"Video index (1-based): {idx}")
    print(f"Time window for annotate_videos_with_sleap_and_trials: (\"{_fmt_hhmmss(start_offset)}\", \"{_fmt_hhmmss(end_offset)}\")")
    print(f"Video covers {t_min} to {t_max} (clock time)")
    if end_dt > t_max:
        over = (end_dt - t_max).total_seconds()
        print(f"Warning: requested end extends {over:.2f}s past this video; window clipped to video end")


Video file: 2025-11-28T15-25-28__VideoData_1904-01-03T04-00-00.avi
Video index (1-based): 2
Time window for annotate_videos_with_sleap_and_trials: ("00:47:46", "00:48:17")
Video covers 2025-11-28 15:41:58.005792 to 2025-11-28 16:41:57.994400 (clock time)


In [ ]:
# Inspect Port0 events in a time window (raw digital input) --> get all Poke IN and OUT events in the time window
import pandas as pd
from hypnose_analysis.utils.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 40
date = 20251128  # YYYYMMDD
start_time_str = "15:45:22"  # HH:MM:SS or HH:MM:SS.mmm
end_time_str   = "15:45:23"  # HH:MM:SS or HH:MM:SS.mmm
experiment_index = 0  # if multiple runs exist for the same date
# ----------------------

# Resolve experiment root using the same helper as classification_utils
exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; check subjid/date/index")
print(f"Using experiment root: {exp_root}")

# Load streams (uses heartbeat correction internally); verbose=True to show progress
streams = load_all_streams(exp_root, apply_corrections=True, verbose=True)
port0 = streams.get("digital_input_data", {}).get("DIPort0")
if port0 is None or port0.empty:
    raise ValueError("DIPort0 not found or empty")
port0 = port0.astype(bool).sort_index()

# Define window
start_dt = pd.to_datetime(f"{date} {start_time_str}", errors="coerce")
end_dt = pd.to_datetime(f"{date} {end_time_str}", errors="coerce")
if pd.isna(start_dt) or pd.isna(end_dt):
    raise ValueError("Could not parse start/end times")
if end_dt <= start_dt:
    raise ValueError("end_time must be after start_time")

# Slice and find edges
slice_ser = port0.loc[start_dt:end_dt]
if slice_ser.empty:
    raise ValueError("No Port0 samples in the requested window; adjust times")

rises = slice_ser & ~slice_ser.shift(1, fill_value=False)
falls = ~slice_ser & slice_ser.shift(1, fill_value=False)

rise_times = rises[rises].index.to_list()
fall_times = falls[falls].index.to_list()

print(f"Port0 window: {start_dt} -> {end_dt}")
print(f"N samples in window: {len(slice_ser)}")
print(f"Rising edges (poke-in): {len(rise_times)}")
for t in rise_times:
    print(f"  IN  @ {t}")
print(f"Falling edges (poke-out): {len(fall_times)}")
for t in fall_times:
    print(f"  OUT @ {t}")


Using subject directory: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259
Loaded experiment 0: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28
Using experiment root: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28
Loading data streams from: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28
Loaded heartbeat data
Calculated real-time offset: 44525 days, 11:41:58
Created timestamp interpolation mapping
Loaded digital_input_data
Loaded output_set
Loaded output_clear
Loaded olfactometer_valves_0
Loaded olfactometer_valves_1
Loaded olfactometer_end_0
Loaded analog_data
Loaded flow_meter
Loaded video_data
Loaded pulse_supply_1
Loaded pulse_supply_2
Created odour_led

Applying time corrections to all data streams...
Applied correction to digital_input_data
Applied correction t

In [ ]:
# Get the nearest Poke OUT for a given timestamp, and the video frame (rawdata) that is closes to that timestamp. 
import pandas as pd
from pathlib import Path
from hypnose_analysis.utils.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 40
date = 20251128
experiment_index = 0  # pick run if multiple
port0_event_ts = "2025-11-28 15:45:22.563520"  # falling edge (clock-corrected)
target_video_name = "VideoData_1904-01-03T04-00-00.avi"  # filter to this AVI; set None to search all
# ----------------------

# Resolve experiment root and load raw streams (device-time + folder offset)
exp_root = load_experiment(subjid, date, index=experiment_index)
streams = load_all_streams(exp_root, apply_corrections=True, verbose=False)

# Port0 falling edges
port0 = streams.get("digital_input_data", {}).get("DIPort0")
if port0 is None or port0.empty:
    raise ValueError("DIPort0 missing")
port0 = port0.astype(bool).sort_index()
falls = (~port0) & port0.shift(1, fill_value=False)
fall_times = falls[falls].index

# Find nearest falling edge to the provided timestamp
event_ts = pd.to_datetime(port0_event_ts, errors="coerce")
if pd.isna(event_ts):
    raise ValueError("Could not parse port0_event_ts")
# TimedeltaIndex lacks .abs; use numpy absolute
nearest_edge = (pd.Index(abs(fall_times - event_ts))).argmin()
edge_ts = fall_times[nearest_edge]
edge_delta_s = (edge_ts - event_ts).total_seconds()
print(f"Requested Port0 OUT: {event_ts}")
print(f"Nearest Port0 OUT:   {edge_ts} (delta {edge_delta_s:+.6f} s)")

# Video metadata
video_df = streams.get("video_data")
if video_df is None or video_df.empty:
    raise ValueError("video_data missing")
video_df = video_df.copy()

# Optional filter to a specific AVI
if target_video_name:
    video_df = video_df[video_df['_path'].astype(str).str.endswith(target_video_name)]
    if video_df.empty:
        raise ValueError(f"No video_data rows for {target_video_name}")

# Ensure hardware columns present
if 'hw_counter' not in video_df.columns:
    raise ValueError("hw_counter column missing in video_data")

# Find nearest frame by timestamp
nearest_frame_idx = (pd.Index(abs(video_df.index - edge_ts))).argmin()
frame_row = video_df.iloc[nearest_frame_idx]
raw_hw_counter = int(frame_row['hw_counter']) if pd.notna(frame_row['hw_counter']) else None
first_hw_counter = int(video_df['hw_counter'].iloc[0]) if pd.notna(video_df['hw_counter'].iloc[0]) else None
corrected_frame_id = raw_hw_counter - first_hw_counter if raw_hw_counter is not None and first_hw_counter is not None else None
time_delta_s = (frame_row.name - edge_ts).total_seconds()

print("\nNearest frame to Port0 OUT:")
print(f"  video path: {frame_row.get('_path')}")
print(f"  frame time: {frame_row.name}")
print(f"  hw_counter (raw): {raw_hw_counter}")
print(f"  hw_counter (corrected to start): {corrected_frame_id}")
print(f"  time delta vs Port0 OUT: {time_delta_s:+.6f} s")
if raw_hw_counter is not None:
    count_inclusive = (video_df['hw_counter'] <= raw_hw_counter).sum()
    print(f"  frames up to and including this hw_counter: {count_inclusive}")



Using subject directory: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259
Loaded experiment 0: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28
Requested Port0 OUT: 2025-11-28 15:45:22.563520
Nearest Port0 OUT:   2025-11-28 15:45:22.563520 (delta +0.000000 s)

Nearest frame to Port0 OUT:
  video path: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28\VideoData\VideoData_1904-01-03T04-00-00.avi
  frame time: 2025-11-28 15:45:22.565184
  hw_counter (raw): 2766130
  hw_counter (corrected to start): 12274
  time delta vs Port0 OUT: +0.001664 s
  frames up to and including this hw_counter: 12275


In [ ]:
# Check clock alignment between Port0 events and video frame timestamps (raw data only). As we record at 60 fps, a perfect match is within ~16.7 ms (thus, tolerance at 17 ms should capture all Port0 matches).
import pandas as pd
from hypnose_analysis.utils.classification_utils import load_all_streams, load_experiment

# --- Configure here ---
subjid = 40
date = 20251128
experiment_index = 0          # pick run if multiple
apply_corrections = True      # use heartbeat/folder offset; set False to stay in device time
tolerance_ms = 17              # max allowed delta for a "match"
# ----------------------

exp_root = load_experiment(subjid, date, index=experiment_index)
if exp_root is None:
    raise ValueError("Could not resolve experiment root; check inputs")
streams = load_all_streams(exp_root, apply_corrections=apply_corrections, verbose=False)

# Port0 timestamps
dip0 = streams.get("digital_input_data", {}).get("DIPort0")
if dip0 is None or dip0.empty:
    raise ValueError("DIPort0 missing or empty")
port0_times = pd.DataFrame({"time_port0": pd.to_datetime(dip0.index)}).dropna().sort_values("time_port0")

# Video frame timestamps
video_df = streams.get("video_data")
if video_df is None or video_df.empty:
    raise ValueError("video_data missing or empty")
video_times = pd.DataFrame({"time_video": pd.to_datetime(video_df.index)}).dropna().sort_values("time_video")

# Merge-asof both directions to check nearest matches within tolerance
tol = pd.Timedelta(milliseconds=tolerance_ms)

v_to_p = pd.merge_asof(video_times, port0_times, left_on="time_video", right_on="time_port0", direction="nearest", tolerance=tol)
p_to_v = pd.merge_asof(port0_times, video_times, left_on="time_port0", right_on="time_video", direction="nearest", tolerance=tol)

unmatched_video = v_to_p["time_port0"].isna().sum()
unmatched_port0 = p_to_v["time_video"].isna().sum()
matched_video = len(v_to_p) - unmatched_video
matched_port0 = len(p_to_v) - unmatched_port0

print(f"Streams loaded from: {exp_root}")
print(f"apply_corrections={apply_corrections}, tolerance={tolerance_ms} ms")
print(f"Video frames: {len(video_times):,}, Port0 samples: {len(port0_times):,}")
print(f"Video frames with match: {matched_video:,} (unmatched: {unmatched_video:,})")
print(f"Port0 samples with match: {matched_port0:,} (unmatched: {unmatched_port0:,})")

# Show first 10 matched pairs (video -> port0) with deltas
matched_rows = v_to_p.dropna(subset=["time_port0"]).head(10).copy()
if not matched_rows.empty:
    matched_rows["delta_ms"] = (matched_rows["time_port0"] - matched_rows["time_video"]).dt.total_seconds() * 1000
    print("\nFirst 10 video->Port0 matches:")
    for _, row in matched_rows.iterrows():
        print(f"video {row['time_video']}  |  port0 {row['time_port0']}  |  delta_ms={row['delta_ms']:.3f}")
else:
    print("No video frames matched within tolerance")

matched_rows_p = p_to_v.dropna(subset=["time_video"]).head(10).copy()
if not matched_rows_p.empty:
    matched_rows_p["delta_ms"] = (matched_rows_p["time_video"] - matched_rows_p["time_port0"]).dt.total_seconds() * 1000
    print("\nFirst 10 Port0->video matches:")
    for _, row in matched_rows_p.iterrows():
        print(f"port0 {row['time_port0']}  |  video {row['time_video']}  |  delta_ms={row['delta_ms']:.3f}")
else:
    print("No Port0 samples matched within tolerance")


Using subject directory: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259
Loaded experiment 0: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28
Streams loaded from: \\ceph-gw02.hpc.swc.ucl.ac.uk\harris\hypnose\rawdata\sub-040_id-259\ses-025_date-20251128\behav\2025-11-28T15-25-28
apply_corrections=True, tolerance=17 ms
Video frames: 344,379, Port0 samples: 8,784
Video frames with match: 16,315 (unmatched: 328,064)
Port0 samples with match: 8,784 (unmatched: 0)

First 10 video->Port0 matches:
video 2025-11-28 15:26:11.806304  |  port0 2025-11-28 15:26:11.811136  |  delta_ms=4.832
video 2025-11-28 15:26:11.822976  |  port0 2025-11-28 15:26:11.811136  |  delta_ms=-11.840
video 2025-11-28 15:26:15.972832  |  port0 2025-11-28 15:26:15.978976  |  delta_ms=6.144
video 2025-11-28 15:26:15.989504  |  port0 2025-11-28 15:26:15.978976  |  delta_ms=-10.528
video 2025-11-28 15:26:16.006176  |  port0 2025-11-28 15: